In [ ]:
# Import libraries
from pathlib import Path
import pandas as pd
import time
import numpy as np
import math
import random

#------------------Parâmetros a serem ajustados------------------

REPO_ROOT = Path('/home/guilherme/Documentos/GitHub/ironstone')
INSTANCES_DIR = REPO_ROOT / 'Instances'

INSTANCE_VERSION = 'V19'
FAIXA = '18h'

#----------------------------------------------------------------

print(f'Instância: {INSTANCE_VERSION}')
print(f'Faixa horária: {FAIXA}')

In [ ]:
# Retrieve the input data and load the displacement matrices from GraphGeneration.ipynb
def carregar_dados_instancia(version):
    return {
        'df_cars': pd.read_excel(INSTANCES_DIR / f'Instances {version}_cars.xlsx'),
        'df_occurrences': pd.read_excel(INSTANCES_DIR / f'Instances {version}_occurrences.xlsx'),
        'df_times': pd.read_excel(INSTANCES_DIR / f'Instances {version}_times.xlsx'),
        'df_availability': pd.read_excel(INSTANCES_DIR / f'Instances {version}_availability.xlsx'),
        'df': pd.read_excel(INSTANCES_DIR / f'Instances {version}.xlsx'),
    }


_dados = carregar_dados_instancia(INSTANCE_VERSION)
df_cars = _dados['df_cars']
df_occurrences = _dados['df_occurrences']
df_times = _dados['df_times']
df_availability = _dados['df_availability']
df = _dados['df']


def carregar_matriz(version, faixa=None):    
    aliases = [version]
    if len(version) > 1 and version[0] in 'Vv' and version[1:].isdigit():
        aliases.append(version[1:])

    candidatos = []
    if faixa:
        sufixo = faixa.strip()
        for alias in aliases:
            candidatos.append(INSTANCES_DIR / f'matrix{alias}_{sufixo}.txt')
            candidatos.append(INSTANCES_DIR / f'matrix{alias}_{sufixo.lower()}.txt')
    for alias in aliases:
        candidatos.append(INSTANCES_DIR / f'matrix{alias}.txt')
    for alias in aliases:
        candidatos.extend(sorted(INSTANCES_DIR.glob(f'matrix{alias}_*.txt')))

    for path in candidatos:
        if path.exists():
            return eval(path.read_text(encoding='utf-8'))

    raise FileNotFoundError(
        f'Nenhuma matriz foi encontrada para {version} em {INSTANCES_DIR}.'
    )


matriz = carregar_matriz(INSTANCE_VERSION, FAIXA)

n_pts = len(matriz)
print(f'Dados: {len(df)} pontos')
print(f'Matriz: {n_pts}x{n_pts}') 

In [ ]:
# Initialize the Solution Matrix with the sequence of service locations.
def ini_solucao_seq_atendimento(df_cars): 
    solucao_seq_atendimento = []
    linha1 = []
    for i in range(len(df_cars)):
        list.append(linha1, [])
    solucao_seq_atendimento = solucao_seq_atendimento + linha1     
    
    return solucao_seq_atendimento


# Initialize the Solution Matrix with the service times.
def ini_solucao_t_atendimento(df_cars): 
    solucao_t_atendimento = []
    linha2 = []
    for i in range(len(df_cars)):
        list.append(linha2, [])
    solucao_t_atendimento = solucao_t_atendimento + linha2     
    
    return solucao_t_atendimento


#Initialize the vector of occurrence cardinality: 0 -> not served / 1 -> served.
def ini_card_ocorrencias(df_occurrences):
    card_ocorrencias = []
    for i in df_occurrences['Unid']:
        atendido = 0
        card_ocorrencias.append(atendido)
    
    return card_ocorrencias


#Initialize the vector of occurrence priorities based on the input data.
def ini_prio_ocorrencias(df_times):
    prio_ocorrencias = []
    for i, p in enumerate (df_times['Prioridade']):
        prioridade = df_times.loc[i, "Prioridade"]
        prio_ocorrencias.append(prioridade)
    
    return prio_ocorrencias


#Initialize the vector of occurrence durations based on the input data.
def ini_duracao(df_times):
    duracao = []
    for i, d in enumerate (df_times['Duração']):
        du = df_times.loc[i, "Duração"]
        duracao.append(du)
    
    return duracao


#Initialize the initial position of the service units in the Displacement Matrix.
def ini_pos_k(df_availability):
    pos_k = []
    for i, p in enumerate (df_availability['Unid']):
        posicao = i
        pos_k.append(posicao)
    
    return pos_k


#Initialize the vector of service unit availability based on the input data.
def ini_v_disponib_k(df_availability):
    v_disponib_k = []
    for i, d in enumerate (df_availability['Diponib']):
        disponibilidade = df_availability.loc[i, 'Diponib']
        v_disponib_k.append(disponibilidade)
    
    return v_disponib_k


#Add the chosen candidate in the Solution Matrix with the sequence of service locations.
def atu_solucao_seq_atendimento(solucao_seq_atendimento, escolha):
    solucao_seq_atendimento[escolha[0]-1].append(escolha[1])


#Add the chosen candidate in the Solution Matrix with the service time.
def atu_solucao_t_atendimento(solucao_t_atendimento, escolha):
    solucao_t_atendimento[escolha[0]-1].append(escolha[2])


#Update the value of the solution.
def atu_valor_da_solucao(solucao_t_atendimento):
    valor_da_solucao = 0
    for i in solucao_t_atendimento:
        if len(i) > 0:
            a = i[-1]
            if a > valor_da_solucao:
                valor_da_solucao = a
        else:
            a = 0
            valor_da_solucao += a
    
    return valor_da_solucao


#Update the vector of occurrence cardinality.
def atu_card_ocorrencias(card_ocorrencias, escolha):
    card_ocorrencias[escolha[1]-1] = 1


#Update the vector of occurrence priorities.
def atu_prio_ocorrencias(prio_ocorrencias, escolha):
    prio_ocorrencias[escolha[1]-1] = 100


#Update the vector of service unit positions.
def atu_pos_k(pos_k, escolha, v_disponib_k):
    pos_k[escolha[0]-1] = len(v_disponib_k) + escolha[1]-1


#Update the vector of service unit availability.
def atu_v_disponib_k(v_disponib_k, escolha):
    v_disponib_k[escolha[0]-1] = escolha[2]


# Choose the candidate with an alpha degree of randomness.
def escolher_candidado_alfa(candidatos_ordem, alfa, escolha_min, df_cars):
    if len(candidatos_ordem) == len(df_cars):
        escolha = candidatos_ordem[0]
    else:
        tam_cand = len(candidatos_ordem)-1
        max_cand = math.floor(tam_cand * alfa)
        if max_cand < escolha_min:
            max_cand = escolha_min
        esc = random.randint(0, max_cand)
        escolha = candidatos_ordem[esc]
    
    return escolha


#Choose the candidate with the lowest completion time.
def escolher_candidato(candidatos):
    t_menor = 1000
    for i in candidatos:
        if i[2] < t_menor:
            t_menor = i[2]
            escolha = i
    
    return escolha

In [ ]:
def gerar_candidatos(v_disponib_k, duracao, prio_ocorrencias, pos_k, card_ocorrencias, matriz):
    candidatos = []
    n_carros = len(v_disponib_k)
    prio_min = min(prio_ocorrencias)
    for i, disponibilidade in enumerate(v_disponib_k):
        c = i + 1
        linha = []
        for j, dur in enumerate(duracao):
            if card_ocorrencias[j] == 0 and prio_ocorrencias[j] == prio_min:
                o = j + 1
                desloc = matriz[pos_k[i]][j + n_carros]
                t = round(disponibilidade + dur + desloc, 1)
                linha.append([c, o, t])
        candidatos += linha
    
    return candidatos


#Sort the Candidates Matrix.
def ordenar_candidatos(candidatos):
    candidatos_ordem = np.array(candidatos)
    candidatos_ordem = candidatos_ordem[candidatos_ordem[:,2].argsort()]
    candidatos_ordem = candidatos_ordem.tolist()
    for i in candidatos_ordem:
        i[0] = int(i[0])
        i[1] = int(i[1])
        i[2] = float(i[2])
    
    return candidatos_ordem

In [ ]:
def avaliar(atribuicao_por_veiculo, num_veiculos, disponibilidade_inicial, posicao_inicial, prioridades, duracoes):
    rotas_ordenadas = []
    tempos_fim_por_veiculo = []
    makespan = 0.0
    soma_tempos_finais = 0.0

    for ind_veiculo in range(num_veiculos):
        # ordena as ocorrências de cada veículo pela prioridade (ordenação estável)
        rota = sorted(
            atribuicao_por_veiculo[ind_veiculo],
            key=lambda id_ocorrencia: prioridades[id_ocorrencia - 1],
        )

        minuto_atual = disponibilidade_inicial[ind_veiculo]
        indice_matriz = posicao_inicial[ind_veiculo]

        # armazena os tempos de término de cada ocorrência
        tempos_fim_rota = []

        for id_ocorrencia in rota:
            indice_destino = (id_ocorrencia - 1) + num_veiculos
            desloc = matriz[indice_matriz][indice_destino]
            
            minuto_atual = round(minuto_atual + duracoes[id_ocorrencia - 1] + desloc, 1)
            tempos_fim_rota.append(minuto_atual)
            indice_matriz = indice_destino

        rotas_ordenadas.append(rota)
        tempos_fim_por_veiculo.append(tempos_fim_rota)

        if tempos_fim_rota:
            tempo_fim_veiculo = tempos_fim_rota[-1]
            soma_tempos_finais += tempo_fim_veiculo
            if tempo_fim_veiculo > makespan:
                makespan = tempo_fim_veiculo

    soma_tempos_finais = round(soma_tempos_finais, 1)
    return rotas_ordenadas, tempos_fim_por_veiculo, makespan, soma_tempos_finais


def posicoes_validas_insercao(rota_destino, id_ocorrencia, prioridades):
    """
    Retorna os índices válidos de inserção dentro do bloco de mesma prioridade
    da rota destino.
    """
    
    prio_ocorrencia = prioridades[id_ocorrencia - 1]

    # encontra o início do bloco de mesma prioridade
    pos_inicio = 0
    for oc in rota_destino:
        if prioridades[oc - 1] < prio_ocorrencia:
            pos_inicio += 1
        else:
            break

    # avança até o fim do bloco de mesma prioridade
    pos_fim = pos_inicio
    for idx in range(pos_inicio, len(rota_destino)):
        if prioridades[rota_destino[idx] - 1] == prio_ocorrencia:
            pos_fim += 1
        else:
            break

    return range(pos_inicio, pos_fim + 1)


def eh_melhor_solucao(makespan_vizinho, soma_vizinho, makespan_atual, soma_atual):
    if makespan_vizinho < makespan_atual:
        return True
    if makespan_vizinho == makespan_atual and soma_vizinho < soma_atual:
        return True
    return False


def avaliar_rotas_alteradas(rotas_candidatas, veiculos_alterados, tempos_fim_por_veiculo,
                            num_veiculos, disponibilidade_inicial, posicao_inicial,
                            prioridades, duracoes):

    tempos_vizinhos = list(tempos_fim_por_veiculo)

    for ind_veiculo in veiculos_alterados:
        # ordena as ocorrências do veículo pela prioridade
        rota = sorted(
            rotas_candidatas[ind_veiculo],
            key=lambda id_ocorrencia: prioridades[id_ocorrencia - 1],
        )
        rotas_candidatas[ind_veiculo] = rota

        minuto_atual = disponibilidade_inicial[ind_veiculo]
        indice_matriz = posicao_inicial[ind_veiculo]

        # armazena os tempos de término de cada ocorrência
        tempos_fim_rota = []

        for id_ocorrencia in rota:
            indice_destino = (id_ocorrencia - 1) + num_veiculos
            desloc = matriz[indice_matriz][indice_destino]

            minuto_atual = round(minuto_atual + duracoes[id_ocorrencia - 1] + desloc, 1)
            tempos_fim_rota.append(minuto_atual)
            indice_matriz = indice_destino

        tempos_vizinhos[ind_veiculo] = tempos_fim_rota

    makespan = 0.0
    soma_tempos_finais = 0.0

    for tempos_rota in tempos_vizinhos:
        if tempos_rota:
            tempo_fim_veiculo = tempos_rota[-1]
            soma_tempos_finais += tempo_fim_veiculo
            if tempo_fim_veiculo > makespan:
                makespan = tempo_fim_veiculo

    soma_tempos_finais = round(soma_tempos_finais, 1)
    return rotas_candidatas, tempos_vizinhos, makespan, soma_tempos_finais


def busca_local(atribuicao_por_veiculo, num_veiculos=None, disponibilidade_inicial=None, posicao_inicial=None, prioridades=None, duracoes=None):
    # Aceita a primeira melhoria, com desempate pela soma dos tempos totais.
    
    if num_veiculos is None:
        num_veiculos = len(df_availability)
    
    if disponibilidade_inicial is None:
        disponibilidade_inicial = ini_v_disponib_k(df_availability)
    
    if posicao_inicial is None:
        posicao_inicial = ini_pos_k(df_availability)
    
    if prioridades is None:
        prioridades = ini_prio_ocorrencias(df_times)
    
    if duracoes is None:
        duracoes = ini_duracao(df_times)

    
    rotas, tempos_fim_por_veiculo, makespan_atual, soma_atual = avaliar(
        atribuicao_por_veiculo, num_veiculos, disponibilidade_inicial, 
        posicao_inicial, prioridades, duracoes,
    )
    
    numero_rotas = len(rotas)

    houve_melhoria = True
    while houve_melhoria:  
        houve_melhoria = False
        veiculo_gargalo = 0
        maior_tempo_fim = 0.0
        
        # para cada carro e seus tempos de fim de rota
        for ind_veiculo, tempos_rota in enumerate(tempos_fim_por_veiculo):
            if not tempos_rota:
                continue

            tempo_fim = tempos_rota[-1]

            if tempo_fim > maior_tempo_fim:
                maior_tempo_fim = tempo_fim
                veiculo_gargalo = ind_veiculo

        # mover uma ocorrencia do gargalo para outro veiculo
        # testando todas as posicoes validas dentro do bloco de mesma prioridade
        for id_ocorrencia in rotas[veiculo_gargalo]:
            for ind_veiculo_destino in range(numero_rotas):
                if ind_veiculo_destino == veiculo_gargalo:
                    continue

                rota_destino_atual = rotas[ind_veiculo_destino]
                posicoes_insercao = posicoes_validas_insercao(
                    rota_destino_atual, id_ocorrencia, prioridades
                )

                for pos_insercao in posicoes_insercao:
                    # copia das rotas (cada rota, nao so a lista externa)
                    rotas_candidatas = [rota.copy() for rota in rotas]

                    # remove a ocorrencia do gargalo
                    rotas_candidatas[veiculo_gargalo].remove(id_ocorrencia)

                    # insere a ocorrencia na posicao valida do veiculo destino
                    rotas_candidatas[ind_veiculo_destino].insert(pos_insercao, id_ocorrencia)

                    rotas_vizinhas, tempos_vizinhos, makespan_vizinho, soma_vizinho = avaliar_rotas_alteradas(
                        rotas_candidatas, [veiculo_gargalo, ind_veiculo_destino], tempos_fim_por_veiculo,
                        num_veiculos, disponibilidade_inicial, posicao_inicial, prioridades, duracoes,
                    )
                    
                    # se encontrei solucao melhor na vizinhanca
                    if eh_melhor_solucao(makespan_vizinho, soma_vizinho, makespan_atual, soma_atual):
                        rotas = rotas_vizinhas
                        tempos_fim_por_veiculo = tempos_vizinhos
                        makespan_atual = makespan_vizinho
                        soma_atual = soma_vizinho
                        houve_melhoria = True
                        break

                if houve_melhoria:
                    break

            if houve_melhoria:
                break

        if houve_melhoria:
            continue

        # trocar uma ocorrencia do gargalo por uma de outro veiculo
        for id_ocorrencia in rotas[veiculo_gargalo]:

            # para cada veiculo
            for ind_veiculo_destino in range(numero_rotas):
                if ind_veiculo_destino == veiculo_gargalo:
                    continue
                
                # para cada ocorrencia do novo veiculo
                for id_ocorrencia_troca in rotas[ind_veiculo_destino]:

                    # copia das rotas
                    rotas_candidatas = [rota.copy() for rota in rotas]
                    
                    # remove a ocorrencia do gargalo
                    rotas_candidatas[veiculo_gargalo].remove(id_ocorrencia)

                    # adiciona a ocorrencia do gargalo no novo veiculo
                    rotas_candidatas[ind_veiculo_destino].append(id_ocorrencia)

                    # remove a ocorrencia do novo veiculo
                    rotas_candidatas[ind_veiculo_destino].remove(id_ocorrencia_troca)

                    # adiciona a ocorrencia do novo veiculo no gargalo
                    rotas_candidatas[veiculo_gargalo].append(id_ocorrencia_troca)

                    rotas_vizinhas, tempos_vizinhos, makespan_vizinho, soma_vizinho = avaliar_rotas_alteradas(
                        rotas_candidatas, [veiculo_gargalo, ind_veiculo_destino], tempos_fim_por_veiculo,
                        num_veiculos, disponibilidade_inicial, posicao_inicial, prioridades, duracoes,
                    )

                    if eh_melhor_solucao(makespan_vizinho, soma_vizinho, makespan_atual, soma_atual):
                        rotas = rotas_vizinhas
                        tempos_fim_por_veiculo = tempos_vizinhos
                        makespan_atual = makespan_vizinho
                        soma_atual = soma_vizinho
                        houve_melhoria = True
                        break

                if houve_melhoria:
                    break

            if houve_melhoria:
                break

        if houve_melhoria:
            continue

        # trocar a posição de duas ocorrências de mesma prioridade dentro do gargalo
        rota_gargalo = rotas[veiculo_gargalo]
        num_ocorrencias_gargalo = len(rota_gargalo)
        
        for pos_a in range(num_ocorrencias_gargalo):
            for pos_b in range(pos_a + 1, num_ocorrencias_gargalo):
                if prioridades[rota_gargalo[pos_a] - 1] != prioridades[rota_gargalo[pos_b] - 1]:
                    continue

                # copia das rotas
                rotas_candidatas = [rota.copy() for rota in rotas]
                rota_trocada = rotas_candidatas[veiculo_gargalo]
                rota_trocada[pos_a], rota_trocada[pos_b] = rota_trocada[pos_b], rota_trocada[pos_a]

                rotas_vizinhas, tempos_vizinhos, makespan_vizinho, soma_vizinho = avaliar_rotas_alteradas(
                    rotas_candidatas, [veiculo_gargalo], tempos_fim_por_veiculo,
                    num_veiculos, disponibilidade_inicial, posicao_inicial, prioridades, duracoes,
                )
                
                if eh_melhor_solucao(makespan_vizinho, soma_vizinho, makespan_atual, soma_atual):
                    rotas = rotas_vizinhas
                    tempos_fim_por_veiculo = tempos_vizinhos
                    makespan_atual = makespan_vizinho
                    soma_atual = soma_vizinho
                    houve_melhoria = True
                    break

            if houve_melhoria:
                break
            
    return rotas, tempos_fim_por_veiculo

In [ ]:

t0 = time.time()
n = 1
cont = 0
best_valor_da_solucao = 999999

for i in range(n):
    #Initialize the input variables.
    solucao_seq_atendimento = ini_solucao_seq_atendimento(df_cars)
    solucao_t_atendimento = ini_solucao_t_atendimento(df_cars)
    card_ocorrencias = ini_card_ocorrencias(df_occurrences)
    prio_ocorrencias = ini_prio_ocorrencias(df_times)
    duracao = ini_duracao(df_times)
    pos_k = ini_pos_k(df_availability)
    v_disponib_k = ini_v_disponib_k(df_availability)
    valor_da_solucao = 0

    #Loop to enumerate the candidates and put them in the Solution Matrix.
    for i in range (len(card_ocorrencias)):
        candidatos = gerar_candidatos(v_disponib_k, duracao, prio_ocorrencias, pos_k, card_ocorrencias, matriz)
        escolha = escolher_candidato(candidatos)
        atu_solucao_seq_atendimento(solucao_seq_atendimento, escolha)
        atu_solucao_t_atendimento(solucao_t_atendimento, escolha)
        atu_card_ocorrencias(card_ocorrencias, escolha)
        atu_prio_ocorrencias(prio_ocorrencias, escolha)
        atu_pos_k(pos_k, escolha, v_disponib_k)
        atu_v_disponib_k(v_disponib_k, escolha)

    valor_da_solucao = atu_valor_da_solucao(solucao_t_atendimento)
    
    if valor_da_solucao < best_valor_da_solucao:
        cont += 1
        best_solucao_seq_atendimento = solucao_seq_atendimento
        best_solucao_t_atendimento = solucao_t_atendimento
        best_valor_da_solucao = valor_da_solucao
    
print('Greedy:')
print(best_solucao_seq_atendimento)
print(best_solucao_t_atendimento)
print(round(best_valor_da_solucao,1))
print(cont)
tf = time.time()
tempo_execucao = tf - t0
print(f'It took {(tempo_execucao):.2f}s to run')



In [ ]:
def executar_heuristica(
    alfa_ini=0.2,
    incremento_alfa=0.1,
    max_it_sem_melhora_alfa=30,
    max_iteracoes=4612,
    alfa_max=0.5,
    limite_tempo_s=60.0,
    usar_busca_local=False,
    imprimir_infos=True,
):
    t0 = t1 = time.time()
    best_valor_da_solucao = 999999
    best_solucao_seq_atendimento = []
    best_solucao_t_atendimento = []

    it_sem_melhora_alfa = 0
    alfa = alfa_ini
    it = 0

    num_veiculos_bl = len(df_availability)
    disponibilidade_inicial_bl = ini_v_disponib_k(df_availability)
    posicao_inicial_bl = ini_pos_k(df_availability)
    prioridades_bl = ini_prio_ocorrencias(df_times)
    duracoes_bl = ini_duracao(df_times)

    while it < max_iteracoes:
        if t1 - t0 > limite_tempo_s:
            if imprimir_infos:
                print(f'**** Time exceeds {limite_tempo_s:.0f} seconds. ****')
            break

        # Iniciar as variáveis de entrada
        solucao_seq_atendimento = ini_solucao_seq_atendimento(df_cars)
        solucao_t_atendimento = ini_solucao_t_atendimento(df_cars)
        card_ocorrencias = ini_card_ocorrencias(df_occurrences)
        prio_ocorrencias = ini_prio_ocorrencias(df_times)
        duracao = ini_duracao(df_times)
        pos_k = ini_pos_k(df_availability)
        v_disponib_k = ini_v_disponib_k(df_availability)

        # Loop para elencar os candidatos e colocar na matriz de solução
        num_ocorrencias = len(card_ocorrencias)
        for j in range(num_ocorrencias):
            candidatos = gerar_candidatos(v_disponib_k, duracao, prio_ocorrencias, pos_k, card_ocorrencias, matriz)
            candidatos_ordem = ordenar_candidatos(candidatos)

            escolha = escolher_candidado_alfa(candidatos_ordem, alfa, 0, df_cars)

            atu_solucao_seq_atendimento(solucao_seq_atendimento, escolha)
            atu_solucao_t_atendimento(solucao_t_atendimento, escolha)
            atu_card_ocorrencias(card_ocorrencias, escolha)
            atu_prio_ocorrencias(prio_ocorrencias, escolha)
            atu_pos_k(pos_k, escolha, v_disponib_k)
            atu_v_disponib_k(v_disponib_k, escolha)

        if usar_busca_local:
            solucao_seq_atendimento, solucao_t_atendimento = busca_local(
                solucao_seq_atendimento,
                num_veiculos_bl,
                disponibilidade_inicial_bl,
                posicao_inicial_bl,
                prioridades_bl,
                duracoes_bl,
            )

        valor_da_solucao = atu_valor_da_solucao(solucao_t_atendimento)

        if valor_da_solucao < best_valor_da_solucao:
            if imprimir_infos:
                print(f'melhora  f={valor_da_solucao:.1f}  α={alfa:.2f}  it={it}')

            best_solucao_seq_atendimento = solucao_seq_atendimento
            best_solucao_t_atendimento = solucao_t_atendimento
            best_valor_da_solucao = valor_da_solucao

            alfa = alfa_ini
            it = 0
            it_sem_melhora_alfa = 0
        else:
            it_sem_melhora_alfa += 1
            it += 1

        if it_sem_melhora_alfa >= max_it_sem_melhora_alfa:
            alfa = min(alfa_max, alfa + incremento_alfa)
            if imprimir_infos:
                print(f'α → {alfa:.2f}  (it={it})')
            it_sem_melhora_alfa = 0

        t1 = time.time()

    tf = time.time()
    if imprimir_infos:
        print()
        print('IRONSTONE SOLUTION:')
        print('---' * 40)
        print(f'The sequence of occourences held is given by: {best_solucao_seq_atendimento}')
        print(f'The sequence of attendence times is given by: {best_solucao_t_atendimento}')
        print(f'The value of the solution is: {best_valor_da_solucao:.1f}')
        print(f'It took {(tf - t0):.2f}s to run the heuristic')

    return best_valor_da_solucao, best_solucao_seq_atendimento, best_solucao_t_atendimento


#random.seed(0)

best_valor_da_solucao, best_solucao_seq_atendimento, best_solucao_t_atendimento = executar_heuristica(
    alfa_ini=0.25,
    incremento_alfa=0.1,
    max_it_sem_melhora_alfa=30,
    max_iteracoes=350,
    alfa_max=0.5,
    limite_tempo_s=60.0,
    usar_busca_local=True,
)


"""
best_valor_da_solucao, best_solucao_seq_atendimento, best_solucao_t_atendimento = executar_heuristica(
    alfa_ini=0.2,
    incremento_alfa=0.1,
    max_it_sem_melhora_alfa=1048,
    max_iteracoes=4612,
    alfa_max=0.5,
    limite_tempo_s=60.0,
    usar_busca_local=False,
)
"""

